# 🔬 SkinGuardAI: MedFusionNet Training on College HPC (Jupyter Lab)

This notebook trains the state-of-the-art **MedFusionNet** architecture (ConvNeXt-Tiny + Pretrained Swin Transformer) on the **HAM10000** skin lesion dataset from scratch.

### Key Upgrades Included:
- **Dual Pretrained Backbones**: ImageNet-1K pretrained ConvNeXt-Tiny + Swin-Transformer Tiny.
- **AdamW & Cosine Annealing**: Linear warmup + Cosine learning rate decay to eliminate accuracy oscillations.
- **Advanced Augmentation**: RandomErasing, Affine, Jitter to prevent overfitting.
- **Label Smoothing (0.1)**: Softens borderline lesion targets for >95% generalization accuracy.
- **Early Stopping**: Halts automatically after 10 epochs without validation improvement.
- **Bayesian Hyperparameter Tuning**: Automated search with Optuna.
- **Clean Visual Progress**: Formatted epoch cards without terminal clutter.

## 1. Verify GPU Hardware and CUDA Environment

In [ ]:
!nvidia-smi

import torch
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:", torch.cuda.get_device_name(0))
    print("Total VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), "GB")
else:
    print("⚠️ WARNING: CUDA not detected! Check your Jupyter kernel / CUDA modules.")

## 2. Verify Dataset Splits
Ensure `dataset/` and `splits/` are in the current working directory.

In [ ]:
import os
import pandas as pd

assert os.path.exists("splits/train.csv"), "Missing splits/train.csv!"
assert os.path.exists("splits/val.csv"), "Missing splits/val.csv!"
assert os.path.exists("splits/test.csv"), "Missing splits/test.csv!"

train_df = pd.read_csv("splits/train.csv")
val_df = pd.read_csv("splits/val.csv")
test_df = pd.read_csv("splits/test.csv")

print(f"Dataset verified:")
print(f" - Train: {len(train_df):,} images")
print(f" - Val:   {len(val_df):,} images")
print(f" - Test:  {len(test_df):,} images")

## 3. (Optional) Automated Bayesian Hyperparameter Tuning with Optuna
Runs 15 quick trials with automated pruning to find the optimal learning rate, weight decay, label smoothing, and dropout.

In [ ]:
# Run Optuna Bayesian tuning (adjust --n-trials as desired)
!python tune.py \
  --n-trials 15 \
  --epochs-per-trial 8 \
  --batch-size 16 \
  --accumulation-steps 1 \
  --num-workers 4
  --vit-type swin

## 4. Train MedFusionNet from Scratch with Early Stopping
On high-memory HPC GPUs (e.g. A100, V100, RTX 3090/4090, T4), you can use `--batch-size 16` or `32`.

In [ ]:
# Launch state-of-the-art training with Gated Fusion, 384x384 Resolution, Focal Loss, Balanced Sampler & Mixup
!python train.py \
  --epochs 50 \
  --batch-size 16 \
  --image-size 384 \
  --vit-type swin \
  --optimizer adamw \
  --scheduler cosine \
  --warmup-epochs 3 \
  --learning-rate 2e-4 \
  --backbone-lr 2e-5 \
  --weight-decay 1e-2 \
  --loss focal \
  --focal-gamma 2.0 \
  --label-smoothing 0.05 \
  --balance balanced \
  --mixup 0.20 \
  --tta \
  --best-metric accuracy \
  --patience 10


## 5. Plot Training & Validation Curves
Visualize loss, top-1 accuracy, and macro-F1 over epochs.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

history_path = "outputs/training_history.csv"
if os.path.exists(history_path):
    df = pd.read_csv(history_path)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # Accuracy curve
    ax1.plot(df["epoch"], df["train_accuracy"] * 100, label="Train Accuracy", marker="o", linewidth=2)
    ax1.plot(df["epoch"], df["val_accuracy"] * 100, label="Val Accuracy", marker="s", linewidth=2)
    ax1.set_title("Top-1 Classification Accuracy (%)", fontsize=13, fontweight="bold")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Accuracy (%)")
    ax1.grid(True, linestyle="--", alpha=0.5)
    ax1.legend()

    # Loss curve
    ax2.plot(df["epoch"], df["train_loss"], label="Train Loss", marker="o", linewidth=2)
    ax2.plot(df["epoch"], df["val_loss"], label="Val Loss", marker="s", linewidth=2)
    ax2.set_title("Cross-Entropy Loss", fontsize=13, fontweight="bold")
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("Loss")
    ax2.grid(True, linestyle="--", alpha=0.5)
    ax2.legend()

    plt.tight_layout()
    plt.show()
else:
    print("No history CSV found yet.")

## 6. Evaluate Best Model on the Unseen Test Set
Generates confusion matrix, multi-class ROC curves, and detailed classification report.

In [ ]:
!python evaluate.py --checkpoint checkpoints/medfusionnet_best.pt

# Display generated Confusion Matrix and ROC Curves inline
from IPython.display import Image, display

cm_path = "outputs/evaluation/confusion_matrix.png"
roc_path = "outputs/evaluation/roc_curves.png"

if os.path.exists(cm_path):
    print("\n--- Confusion Matrix ---")
    display(Image(cm_path, width=550))

if os.path.exists(roc_path):
    print("\n--- ROC Curves ---")
    display(Image(roc_path, width=550))

## 7. Archive Trained Checkpoints for Download
Compresses `checkpoints/medfusionnet_best.pt` and evaluation outputs into a zip file for download to your laptop.

In [ ]:
!zip -r best_model_bundle.zip checkpoints/medfusionnet_best.pt outputs/
print("Bundle created: best_model_bundle.zip. You can now download it directly from Jupyter Lab file browser!")